In [1]:
#Sheet PTC
import pandas as pd

In [2]:
PTC_df = pd.read_excel("Weekly PTA CGL Revised Open Order Report 2025-12-19.xlsx")

In [3]:
print(PTC_df.columns)

Index(['Customer Name', 'Customer Abbreviation', 'Customer Po',
       'Customer Po Line', 'Order date 1', 'Order No', 'Due Date Code',
       'Part No', 'PN used', 'Order.PN', 'PN AL78', 'Description',
       'Revised Due Date', 'ESD', 'Qty Open', 'Unit Price', 'Total Value Open',
       'Qty Resvered', 'Qty Picked', 'Qty Backordered', 'Full/Partial',
       'Mon Reqrd.', 'Mon.Reqrd.2 weeks ago', 'chk', 'Year Reqrd.',
       'Mon ListPO', 'Year 2 weeks ago', 'chk. Year', 'Order date 2',
       'Date Entered', 'Current Sales Status Code', 'Fin Business Code',
       'Carrier Code'],
      dtype='object')


In [4]:
#SORT PN Used A-Z
PTC_df['Order date 1'] = pd.to_datetime(
    PTC_df['Order date 1'],
    errors='coerce'   # invalid dates become NaT
)

#SORT Order Date Oldest to Newest
PTC_df = PTC_df.sort_values(
    by=['PN used', 'Order date 1'],
    ascending=[True, True]
).reset_index(drop=True)


In [5]:
#Make Seqnc.
PTC_df['PN used'] = PTC_df['PN used'].astype(str).str.strip()

# Create sequence per PN used
PTC_df['Seqnc'] = PTC_df.groupby('PN used').cumcount() + 1

# Insert 'Seqnc' after 'Year 2 weeks ago'
insert_pos = PTC_df.columns.get_loc('Year 2 weeks ago') + 1
col = PTC_df.pop('Seqnc')
PTC_df.insert(insert_pos, 'Seqnc', col)


In [6]:
# Create Sqc.PN column
PTC_df['Sqc.PN'] = (
    PTC_df['Seqnc'].astype(str)
    + '.'
    + PTC_df['PN AL78'].astype(str)
)

# Insert 'Sqc.PN' after 'Date Entered'
insert_pos = PTC_df.columns.get_loc('Date Entered') + 1
col = PTC_df.pop('Sqc.PN')
PTC_df.insert(insert_pos, 'Sqc.PN', col)


In [7]:
# Ensure ESD is datetime
PTC_df['ESD'] = pd.to_datetime(PTC_df['ESD'], errors='coerce')

# Create ESD Mon and ESD Year
PTC_df['ESD Mon'] = PTC_df['ESD'].dt.month.astype('Int64')
PTC_df['ESD Year'] = PTC_df['ESD'].dt.year.astype('Int64')

# Insert columns after 'Sqc.PN'
insert_pos = PTC_df.columns.get_loc('Sqc.PN') + 1

col_mon = PTC_df.pop('ESD Mon')
col_year = PTC_df.pop('ESD Year')

PTC_df.insert(insert_pos, 'ESD Mon', col_mon)
PTC_df.insert(insert_pos + 1, 'ESD Year', col_year)


In [8]:
cols_to_int = [
    'Mon.Reqrd.2 weeks ago',
    'Mon ListPO',
    'Year 2 weeks ago'
]

for col in cols_to_int:
    PTC_df[col] = PTC_df[col].astype('Int64')
    
PTC_df = PTC_df.drop(columns=['chk. Year'])

In [9]:
print(PTC_df)

                   Customer Name  Customer Abbreviation Customer Po  \
0     PT. CIPTA ANDALAN TEKNINDO                  16915  M434254208   
1     PT. CIPTA ANDALAN TEKNINDO                  16915  M533254104   
2     PT. CIPTA ANDALAN TEKNINDO                  16915  M433254103   
3     PT. CIPTA ANDALAN TEKNINDO                  16915  M534254213   
4     PT. CIPTA ANDALAN TEKNINDO                  16915  M534254721   
...                          ...                    ...         ...   
2176  PT. CIPTA ANDALAN TEKNINDO                  16915  M534254213   
2177  PT. CIPTA ANDALAN TEKNINDO                 366594  OS33254627   
2178  PT. CIPTA ANDALAN TEKNINDO                  16915  M334254706   
2179  PT. CIPTA ANDALAN TEKNINDO                  16915  M434254709   
2180  PT. CIPTA ANDALAN TEKNINDO                  16915  M534254721   

      Customer Po Line        Order date 1  Order No Due Date Code    Part No  \
0                  161 2025-10-29 10:40:19    759472            S1

In [11]:
from datetime import datetime

today_str = datetime.today().strftime("%Y-%m-%d")
output_file = f"PTC draft {today_str}.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    PTC_df.to_excel(
        writer,
        sheet_name="PTC",
        index=False
    )

print(f"File saved as: {output_file}")


File saved as: PTC draft 2025-12-19.xlsx
